# Berlin Bicycle Traffic Analytics — 2025

Python data exploration, quality checks, transformation, and MySQL/Tableau preparation.

**Data:** Berlin bicycle counts for 2025 and station metadata from the same Excel workbook. **Unit:** hourly bicycle passages per counting station (not distinct cyclists).

**How to run:** Place the source `.xlsx` file in `data/raw/`; launch this notebook from `notebooks/` so `../data/raw` and `../data/processed` resolve correctly. Install `pandas`, `matplotlib`, `openpyxl`, `sqlalchemy`, and `pymysql`. The later database cells require a local MySQL database named `berlin_bicycle_analytics`, appropriate tables, and the `vw_bicycle_traffic` view created separately with the project SQL scripts.


In [ ]:
import pandas as pd

## 1. Inspect source workbook

Identify the official Excel workbook and preview its sheets and station columns.


In [ ]:
from pathlib import Path

data_folder = Path("../data/raw")

excel_files = list(data_folder.glob("*.xlsx"))

excel_files

In [ ]:
if not excel_files:
    raise FileNotFoundError(
        "No Excel file found. Check the data/raw folder."
    )

file_path = excel_files[0]

workbook = pd.ExcelFile(file_path)

print("Available sheets:")
print(workbook.sheet_names)

In [ ]:
df_preview = pd.read_excel(
    file_path,
    sheet_name=0,
    nrows=10
)

display(df_preview)

In [ ]:
df_2025 = pd.read_excel(
    file_path,
    sheet_name="Jahresdatei 2025",
    nrows=10
)

display(df_2025)

In [ ]:
print("Column names:")
print(df_2025.columns.tolist())

print("\nNumber of columns:")
print(df_2025.shape[1])

In [ ]:
df_locations = pd.read_excel(
    file_path,
    sheet_name="Standortdaten",
    nrows=10
)

display(df_locations)

In [ ]:
df_legend = pd.read_excel(
    file_path,
    sheet_name="Legende",
    header=None,
    nrows=20
)

display(df_legend)

In [ ]:
df_2025 = pd.read_excel(
    file_path,
    sheet_name="Jahresdatei 2025",
    nrows=10
)

print("Worksheet loaded: Jahresdatei 2025")
print("Shape:", df_2025.shape)

display(df_2025.iloc[:10, :6])

In [ ]:
print("First column values:")

print(df_2025.iloc[:, 0].tolist())

In [ ]:
df_2025.info()


## 2. Full-year data quality checks

Load the complete 2025 hourly series and inspect missing counts, timestamps, and invalid negative counts. Missing readings are not equivalent to zero bicycle passages.


In [ ]:
df_2025 = pd.read_excel(
    file_path,
    sheet_name="Jahresdatei 2025"
)

print("Total rows:", df_2025.shape[0])
print("Total columns:", df_2025.shape[1])

In [ ]:
missing_values = df_2025.isnull().sum()

print("Total missing values:", missing_values.sum())

display(
    missing_values.sort_values(ascending=False).head(10)
)

In [ ]:
timestamp_col = df_2025.columns[0]

print("Duplicate timestamps:",
      df_2025[timestamp_col].duplicated().sum())

print("First timestamp:", df_2025[timestamp_col].min())
print("Last timestamp:", df_2025[timestamp_col].max())

In [ ]:
station_data = df_2025.iloc[:, 1:]

negative_counts = (station_data < 0).sum().sum()

print("Total negative values:", negative_counts)

## 3. Standardize station columns and examine gaps

Remove installation dates embedded in column headers and visualize missing observations.


In [ ]:
df_clean = df_2025.copy()

df_clean.columns = ["timestamp"] + [
    col.split("\n")[0].strip()
    for col in df_clean.columns[1:]
]

display(df_clean.head())

In [ ]:
missing_by_month = (
    df_clean
    .set_index("timestamp")
    .isna()
    .resample("MS")
    .sum()
)

display(
    missing_by_month[
        ["19-TS-MON", "04-MI-NO", "06-FK-FRA-W"]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

missing_by_month[
    ["19-TS-MON", "04-MI-NO", "06-FK-FRA-W"]
].plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Missing Bicycle Count Observations by Month - 2025")
plt.xlabel("Month")
plt.ylabel("Missing Hours")
plt.tight_layout()
plt.show()

In [ ]:
negative_counts = (df_clean.iloc[:, 1:] < 0).sum().sum()

print("Total negative values:", negative_counts)

## 4. Convert wide records to long format

Transform 8,760 hourly timestamps and 35 counting station columns into 306,600 station-hour rows.


In [ ]:
df_long = df_clean.melt(
    id_vars="timestamp",
    var_name="station_id",
    value_name="bicycle_count"
)

print("Shape:", df_long.shape)

display(df_long.head(10))

In [ ]:
print("Original shape:", df_clean.shape)
print("Long format shape:", df_long.shape)

print("Unique stations:", df_long["station_id"].nunique())

print("Missing counts:", df_long["bicycle_count"].isna().sum())

print(
    "Duplicate station-hour records:",
    df_long.duplicated(
        subset=["timestamp", "station_id"]
    ).sum()
)

In [ ]:
print(
    "Negative bicycle counts:",
    (df_long["bicycle_count"] < 0).sum()
)

## 5. Match station metadata

Read station names and coordinates. Two station IDs differ between sheets (`17-SZ-BRE-O/W` vs `17-SK-BRE-O/W`); standardize them before joining.


In [ ]:
df_locations = pd.read_excel(
    file_path,
    sheet_name="Standortdaten"
)

print("Shape:", df_locations.shape)

display(df_locations.head())

In [ ]:
df_locations = df_locations.rename(columns={
    "Zählstelle": "station_id",
    "Beschreibung - Fahrtrichtung": "station_name",
    "Breitengrad": "latitude",
    "Längengrad": "longitude",
    "Installationsdatum": "installation_date"
})

display(df_locations.head())

In [ ]:
traffic_stations = set(df_long["station_id"])

location_stations = set(df_locations["station_id"])

print("Stations in traffic data:", len(traffic_stations))

print("Stations in location data:", len(location_stations))

print(
    "Stations missing location information:",
    traffic_stations - location_stations
)

print(
    "Duplicate station IDs in metadata:",
    df_locations["station_id"].duplicated().sum()
)

In [ ]:
df_locations[
    df_locations["station_id"]
    .astype(str)
    .str.contains("BRE", case=False, na=False)
]

In [ ]:
traffic_stations = set(df_long["station_id"])
location_stations = set(df_locations["station_id"])

print("In traffic data but not in location data:")
print(traffic_stations - location_stations)

print("\nIn location data but not in traffic data:")
print(location_stations - traffic_stations)

In [ ]:
df_locations["station_id"] = (
    df_locations["station_id"]
    .astype(str)
    .str.strip()
)

df_long["station_id"] = (
    df_long["station_id"]
    .astype(str)
    .str.strip()
)

print(
    "Remaining unmatched traffic stations:",
    set(df_long["station_id"]) -
    set(df_locations["station_id"])
)

In [ ]:
station_id_mapping = {
    "17-SZ-BRE-O": "17-SK-BRE-O",
    "17-SZ-BRE-W": "17-SK-BRE-W"
}

df_long["station_id"] = (
    df_long["station_id"]
    .replace(station_id_mapping)
)

In [ ]:
traffic_stations = set(df_long["station_id"])
location_stations = set(df_locations["station_id"])

print(
    "Unmatched traffic stations:",
    traffic_stations - location_stations
)

print(
    "Unmatched location stations:",
    location_stations - traffic_stations
)

## 6. Merge and validate observations

Join observations to metadata and confirm all 306,600 records are matched.


In [ ]:
df_merged = df_long.merge(
    df_locations,
    on="station_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Merged shape:", df_merged.shape)

display(df_merged.head())

In [ ]:
print("Original rows:", len(df_long))
print("Merged rows:", len(df_merged))

print("\nMerge results:")
print(df_merged["_merge"].value_counts())

print("\nMissing station names:",
      df_merged["station_name"].isna().sum())

print("Missing latitude:",
      df_merged["latitude"].isna().sum())

print("Missing longitude:",
      df_merged["longitude"].isna().sum())

print("Missing bicycle counts:",
      df_merged["bicycle_count"].isna().sum())

In [ ]:
df_merged = df_merged.drop(
    columns="_merge",
    errors="ignore"
)

print("Dataset shape:", df_merged.shape)

## 7. Engineer time features

Create year, month, day, hour, day-of-week, weekend, and season fields.


In [ ]:
df_final = df_merged.copy()

In [ ]:
df_final["year"] = df_final["timestamp"].dt.year

df_final["month"] = df_final["timestamp"].dt.month

df_final["day"] = df_final["timestamp"].dt.day

df_final["hour"] = df_final["timestamp"].dt.hour

df_final["day_of_week"] = (
    df_final["timestamp"].dt.day_name()
)

df_final["weekday_number"] = (
    df_final["timestamp"].dt.dayofweek
)

In [ ]:
df_final["is_weekend"] = (
    df_final["weekday_number"] >= 5
)

In [ ]:
season_mapping = {
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Autumn",
    10: "Autumn",
    11: "Autumn"
}

df_final["season"] = (
    df_final["month"].map(season_mapping)
)

In [ ]:
print("Final shape:", df_final.shape)

print("\nColumn names:")
print(df_final.columns.tolist())

display(df_final.head())

In [ ]:
print("Missing season values:",
      df_final["season"].isna().sum())

print("Missing hour values:",
      df_final["hour"].isna().sum())

print("Unique years:",
      df_final["year"].unique())

print("Unique seasons:",
      df_final["season"].unique())

print("Remaining missing bicycle counts:",
      df_final["bicycle_count"].isna().sum())

## 8. Prepare relational tables and CSV exports

Create station metadata and hourly count tables. Preserve missing bicycle counts (do not impute zero).


In [ ]:
stations = df_locations[
    [
        "station_id",
        "station_name",
        "latitude",
        "longitude",
        "installation_date"
    ]
].copy()

print("Station table shape:", stations.shape)

display(stations.head())

In [ ]:
bicycle_counts = df_final[
    [
        "timestamp",
        "station_id",
        "bicycle_count",
        "year",
        "month",
        "day",
        "hour",
        "day_of_week",
        "weekday_number",
        "is_weekend",
        "season"
    ]
].copy()

print("Bicycle counts shape:", bicycle_counts.shape)

display(bicycle_counts.head())

In [ ]:
print("Unique stations:", stations["station_id"].nunique())

print("Duplicate station IDs:",
      stations["station_id"].duplicated().sum())

print("Total bicycle observations:", len(bicycle_counts))

print("Missing counts:",
      bicycle_counts["bicycle_count"].isna().sum())

print("Missing station IDs:",
      bicycle_counts["station_id"].isna().sum())

print(
    "Duplicate station-hour records:",
    bicycle_counts.duplicated(
        subset=["timestamp", "station_id"]
    ).sum()
)

In [ ]:
from pathlib import Path

output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

stations.to_csv(
    output_folder / "stations.csv",
    index=False
)

bicycle_counts.to_csv(
    output_folder / "bicycle_counts_2025.csv",
    index=False,
    na_rep="\\N",
    date_format="%Y-%m-%d %H:%M:%S"
)

print("CSV files exported successfully!")

In [ ]:
print((output_folder / "stations.csv").exists())

print((output_folder / "bicycle_counts_2025.csv").exists())

## 9. Load records into MySQL

**Prerequisites:** Create the `berlin_bicycle_analytics` database and its `stations` and `bicycle_counts` tables before running these cells. This notebook uses `if_exists="append"`; rerunning uploads can duplicate records or trigger primary-key errors, so import into clean tables or manage existing records deliberately. The password is requested interactively and is not stored in notebook source.


In [ ]:
%pip install sqlalchemy pymysql

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

print("Libraries imported successfully!")

In [ ]:
mysql_user = input("MySQL username: ")
mysql_password = getpass("MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="berlin_bicycle_analytics"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DATABASE()")
    )

    print("Connected to:", result.scalar())

In [ ]:
stations.to_sql(
    name="stations",
    con=engine,
    if_exists="append",
    index=False
)

print("Stations imported successfully!")

In [ ]:
print("Rows:", len(bicycle_counts))

print(
    "Missing bicycle counts:",
    bicycle_counts["bicycle_count"].isna().sum()
)

In [ ]:
bicycle_counts.to_sql(
    name="bicycle_counts",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Bicycle counts imported successfully!")

## 10. Export Tableau-ready dataset

Run the SQL view definition (`vw_bicycle_traffic`) from the project SQL scripts before these cells. Export the joined 2025 dataset for Tableau Public.


In [ ]:
import pandas as pd
from pathlib import Path

query = """
SELECT *
FROM vw_bicycle_traffic
"""

df_dashboard = pd.read_sql(
    query,
    con=engine
)

print("Rows:", len(df_dashboard))
print("Columns:", len(df_dashboard.columns))

display(df_dashboard.head())

In [ ]:
output_folder = Path("../data/processed")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

df_dashboard.to_csv(
    output_folder / "berlin_bicycle_tableau.csv",
    index=False
)

print("Tableau CSV exported successfully!")